In [19]:
%load_ext autoreload

%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import pandas as pd
import numpy as np
import pathlib
import os
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

from final_project_package.ml_logic.model import initialize_model, train_model, evaluate_model
from final_project_package.ml_logic.preprocessor_pipeline import get_fitted_preprocessor


✅ TensorFlow loaded (0.03s)


# Preprocess

In [3]:
path_to_project = "."
split_ratio = 0.3

data_path = pathlib.Path(path_to_project)
data = pd.read_csv(data_path / "data_dump/listings_with_scores.csv")

X = data.drop(columns=["price_man_yen"]).copy()
y = data["price_man_yen"]

def preprocess_y(y):
    return np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=split_ratio)
preprocesser = get_fitted_preprocessor(X_train, y_train)

# Save the preprocessor to a file
filename = data_path / "data_dump/preprocessor.pkl"
with open(filename, 'wb') as file:
    pickle.dump(preprocesser, file)

X_train_preprocessed = pd.DataFrame(preprocesser.transform(X_train), columns=preprocesser.get_feature_names_out(), index=X_train.index)
X_test_preprocessed = pd.DataFrame(preprocesser.transform(X_test), columns=preprocesser.get_feature_names_out(), index=X_test.index)
y_train_preprocessed = pd.Series(preprocess_y(y_train))
y_test_preprocessed = pd.Series(preprocess_y(y_test))

X_train_preprocessed.to_csv(data_path / "data_dump/X_train_preprocessed.csv", index = False)
X_test_preprocessed.to_csv(data_path / "data_dump/X_test_preprocessed.csv", index = False)
y_train_preprocessed.to_csv(data_path / "data_dump/y_train_preprocessed.csv", index = False)
y_test_preprocessed.to_csv(data_path / "data_dump/y_test_preprocessed.csv", index = False)


print(X_train_preprocessed.columns)


Preprocessing features...
✅ returned preprocessor
Index(['keep_rooms__rooms_num', 'num_transformer__area_sqm',
       'num_transformer__year_built', 'num_transformer__floor_number',
       'num_transformer__floors_total', 'num_transformer__walk_minutes',
       'station_transformer__nearest_station',
       'ordinal_transformer__base_layout',
       'mean_luxury_transformer__mean_luxury',
       'mean_brightness_transformer__mean_brightness',
       'mean_condition_transformer__mean_condition'],
      dtype='str')


# Grid Search

In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
#from xgboost import XGBRegressor


## Model with scores

In [5]:
# Load processed data
data_path = pathlib.Path(path_to_project)
X_train = pd.read_csv(data_path / "data_dump/X_train_preprocessed.csv")
y_train = pd.read_csv(data_path / "data_dump/y_train_preprocessed.csv")
X_test = pd.read_csv(data_path / "data_dump/X_test_preprocessed.csv")
y_test = pd.read_csv(data_path / "data_dump/y_test_preprocessed.csv")

# pipeline
pipeline = Pipeline([
    ('regressor', 'passthrough')
])

# Define the parameter grid for different regressors and their hyperparameters
# Note the double underscores (__) used to specify parameters of the named steps

param_grid = [
    # Configuration for Linear Regression
    {
        'regressor': [LinearRegression()],
        # LinearRegression has few hyperparameters, but we could add more preprocessing if needed
    },
    # Configuration for KNN Regression
    {
        'regressor': [KNeighborsRegressor()],
        'regressor__n_neighbors' : range(1, 31)
        # LinearRegression has few hyperparameters, but we could add more preprocessing if needed
    },
    # Configuration for Decision Tree Regression
    {
        'regressor': [DecisionTreeRegressor()],
        'regressor__criterion': ['gini', 'entropy'],
        'regressor__max_depth': [None, 2, 4, 6, 8, 10],
        'regressor__min_samples_split': [2, 5, 10],
        'regressor__min_samples_leaf': [1, 3, 5]
        # LinearRegression has few hyperparameters, but we could add more preprocessing if needed
    },
    # Configuration for Random Forest Regressor
    {
        'regressor': [RandomForestRegressor(random_state=42)],
        'regressor__n_estimators': [50, 100],
        'regressor__max_depth': [None, 10, 20],
    }
]

# The default scoring for regression in GridSearchCV is r2_score
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,                 # Use 5-fold cross-validation
    scoring='neg_mean_squared_error', # A common metric for regression
    verbose=1,            # Output progress
    n_jobs=-1             # Use all available CPU cores
)

grid_search.fit(X_train, y_train)

# 5. Review the results
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation score (negative MSE): {grid_search.best_score_}")

# The best estimator is automatically refitted on the entire training data
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test) # R-squared score
print(f"Test set R-squared score of the best model: {test_score}")

# You can further analyze the results in grid_search.cv_results_

# Save the model to a file
filename = data_path / "data_dump/best_model.sav"
with open(filename, 'wb') as file:
    pickle.dump(best_model, file)

# use our model_evaluate() function

mse = evaluate_model(
    best_model,
    X_test,
    y_test
    )


Fitting 5 folds for each of 145 candidates, totalling 725 fits


/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column

Best parameters found: {'regressor': RandomForestRegressor(random_state=42), 'regressor__max_depth': None, 'regressor__n_estimators': 100}
Best cross-validation score (negative MSE): -0.04568481198042891
Test set R-squared score of the best model: 0.8737551697672103
✅ Model evaluated, MSE: 0.0671987


## Model without the scores

In [6]:
score_columns = ["mean_luxury_transformer__mean_luxury","mean_brightness_transformer__mean_brightness","mean_condition_transformer__mean_condition"]

# Load processed data
data_path = pathlib.Path(path_to_project)
X_train = pd.read_csv(data_path / "data_dump/X_train_preprocessed.csv")
y_train = pd.read_csv(data_path / "data_dump/y_train_preprocessed.csv")
X_test = pd.read_csv(data_path / "data_dump/X_test_preprocessed.csv")
y_test = pd.read_csv(data_path / "data_dump/y_test_preprocessed.csv")

X_train = X_train.drop(columns=score_columns)
X_test = X_test.drop(columns=score_columns)

# pipeline
pipeline = Pipeline([
    ('regressor', 'passthrough')
])

# Define the parameter grid for different regressors and their hyperparameters
# Note the double underscores (__) used to specify parameters of the named steps

param_grid = [
    # Configuration for Linear Regression
    {
        'regressor': [LinearRegression()],
        # LinearRegression has few hyperparameters, but we could add more preprocessing if needed
    },
    # Configuration for KNN Regression
    {
        'regressor': [KNeighborsRegressor()],
        'regressor__n_neighbors' : range(1, 31)
        # LinearRegression has few hyperparameters, but we could add more preprocessing if needed
    },
    # Configuration for Decision Tree Regression
    {
        'regressor': [DecisionTreeRegressor()],
        'regressor__criterion': ['gini', 'entropy'],
        'regressor__max_depth': [None, 2, 4, 6, 8, 10],
        'regressor__min_samples_split': [2, 5, 10],
        'regressor__min_samples_leaf': [1, 3, 5]
        # LinearRegression has few hyperparameters, but we could add more preprocessing if needed
    },
    # Configuration for Random Forest Regressor
    {
        'regressor': [RandomForestRegressor(random_state=42)],
        'regressor__n_estimators': [50, 100],
        'regressor__max_depth': [None, 10, 20],
    }
]

# The default scoring for regression in GridSearchCV is r2_score
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,                 # Use 5-fold cross-validation
    scoring='neg_mean_squared_error', # A common metric for regression
    verbose=1,            # Output progress
    n_jobs=-1             # Use all available CPU cores
)

grid_search.fit(X_train, y_train)

# 5. Review the results
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation score (negative MSE): {grid_search.best_score_}")

# The best estimator is automatically refitted on the entire training data
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test) # R-squared score
print(f"Test set R-squared score of the best model: {test_score}")

# You can further analyze the results in grid_search.cv_results_

# Save the model to a file
filename = data_path / "data_dump/best_model.sav"
with open(filename, 'wb') as file:
    pickle.dump(best_model, file)

# use our model_evaluate() function

mse = evaluate_model(
    best_model,
    X_test,
    y_test
    )


Fitting 5 folds for each of 145 candidates, totalling 725 fits


/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column

Best parameters found: {'regressor': RandomForestRegressor(random_state=42), 'regressor__max_depth': None, 'regressor__n_estimators': 100}
Best cross-validation score (negative MSE): -0.043868224927444154
Test set R-squared score of the best model: 0.8767774490411508
✅ Model evaluated, MSE: 0.06559
